In [ ]:
import os
import numpy as np
import tensorflow as tf

from silx.io.dictdump import h5todict

In [ ]:
cpu = tf.config.list_physical_devices("CPU")[0]
tf.config.set_visible_devices(cpu)

In [ ]:
import matplotlib
from matplotlib import pyplot as plt
COLORS = matplotlib.rcParams['axes.prop_cycle'].by_key()['color']

path_to_figs = "./postproc/testing/"
os.makedirs(path_to_figs, exist_ok=True)


In [ ]:
path_to_model = "./saving/lin_me_int/SavedModel/"
path_to_data = "./data/test.hdf5"
path_to_norm = "./data/norm/"
path_to_edges = "./data/fwd_edge_index.txt"

In [ ]:
model = tf.keras.models.load_model(path_to_model)

In [ ]:
def get_data(data, denorm=True):
    # Collect data
    x = [data[i][1:] for i in ("node_attr", "edge_attr")]
    y = data["target"][1:].squeeze()
    # Denormalize data
    # kf0, na, t = np.split(denormalize(x[1], "edge_attr", denorm), 3, axis=-1)
    kf0, na, t = np.split(x[1], 3, axis=-1)
    kf0, na, t = kf0.squeeze(), na[-1,-1,-1], t[:,-2:-1,-1]
    # Make prediction
    y_pred = model(x).numpy().squeeze()
    kf_pred = model.layers[3].call(x[1]).numpy().squeeze() / (na*t)
    return t.squeeze(), y, y_pred, kf0, kf_pred

In [ ]:
def plot_data(
    x=None,
    y=None,
    xref=None,
    yref=None,
    labels=[r'$t$ [s]', r'$X$'],
    scales=['log', 'log'],
    names=None,
    figname=None
):
    # Initialize figures
    fig = plt.figure()
    ax = fig.add_subplot()
    # x axis
    xlim = [np.amin(x), np.amax(x)]
    ax.set_xlim(xlim)
    ax.set_xlabel(labels[0])
    ax.set_xscale(scales[0])
    # y axis
    ax.set_ylabel(labels[1])
    ax.set_yscale(scales[1])
    # Plotting
    true_style = dict(
        lw=1
    )
    pred_style = dict(
        marker='^',
        fillstyle='none',
        markersize=5,
        markevery=y.shape[0]//20,
        ls='--',
        lw=1
    )
    for d in range(y.shape[1]):
        ax.plot(x, y[:,d], c=COLORS[d], **pred_style)
        ax.plot(xref, yref[:,d], c=COLORS[d], label=names[d], **true_style)
    plt.legend()
    if (figname is not None):
        fig.savefig(figname)
    plt.close()

In [ ]:
data = h5todict(path_to_data)
fwd_edge_index = np.loadtxt(path_to_edges, dtype=int).T

nb_edges = len(fwd_edge_index)
nb_nodes = np.amax(fwd_edge_index) + 1

nodes_names = [r'O$_2^{(%d)}$' % (i+1) for i in range(nb_nodes)]
edges_names = [r'$k_{%d%d}$' % (ij[0]+1,ij[1]+1) for ij in fwd_edge_index]

In [ ]:
for (i, datai) in data.items():
    t, y, y_pred, kf0, kf_pred = get_data(datai, denorm=True)
    kwargs=dict(
        x=t,
        y=y_pred,
        xref=t,
        yref=y,
        labels=[r'$t$ [s]', r'$n$ [m$^{-3}$]'],
        names=nodes_names
    )
    plot_data(
        scales=['log', 'log'],
        figname=path_to_figs + f"/{i}_log",
        **kwargs
    )
    plot_data(
        scales=['log', 'linear'],
        figname=path_to_figs + f"/{i}_lin",
        **kwargs
    )
    kwargs=dict(
        x=t,
        y=kf_pred,
        xref=t,
        yref=kf0,
        labels=[r'$t$ [s]', r'$k_f$ [m$^3$/s]'],
        names=edges_names
    )
    plot_data(
        scales=['log', 'log'],
        figname=path_to_figs + f"/{i}_rates",
        **kwargs
    )